<a href="https://colab.research.google.com/github/hersil7/cdd_ucsc/blob/main/S2_Notebook_Laboratorio_CDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 2 — Calidad y preparación de tu conjunto de datos
**Ciencia de Datos · Semana 2 · Advance UCSC**

| | |
|---|---|
| **Resultado de aprendizaje** | RA1 (cierre) · RA3 (transversal) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Insumo** | El conjunto de datos que elegiste en la semana 1 |
| **Producto** | Conjunto limpio + bitácora completada + avance versionado |
| **Estudiante** | **Hernán Silva** |

---

Este laboratorio **no trae los datos**: los traes tú. Todo lo que hagas aquí es el cuerpo de la **Entrega 1**, que rindes al cierre de esta semana.

Ten abierta en paralelo la planilla `S2_Bitacora_Decisiones_CDD.xlsx`. Escríbela **mientras trabajas**, no al final: reconstruir después por qué tomaste una decisión es mucho más difícil de lo que parece.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata

RUTA = '/content/drive/Othercomputers/Notebook HP/Ing Informática UCSC/Trimestre 3/1 Ciencia de datos/Evaluaciones/Semana 1/Lab 1/'
SEMILLA = 20262
np.random.seed(SEMILLA)

In [ ]:
SEPARADOR = ';'
ARCHIVO = 'ano-2025.csv'

try:
    df = pd.read_csv(RUTA + ARCHIVO, sep=SEPARADOR, encoding='utf-8',
                    low_memory=False, dtype={'fact_anual': str})
    display(df)
except Exception as e:
    print(type(e).__name__)
    print(e)

print('Forma:', df.shape)

,ano_trimestre,mes_central,ano_encuesta,mes_encuesta,region,provincia,tipo,r_p_c,estrato,conglomerado,...,ina_cal_edu,ocup_form,sector,r_p_rev4cl_caenes,tramo_edad,categoria_ocupacion,cae_general,cae_especifico,activ,fact_anual
0,2025,2,2025,1,15,151,1,15101,15000012,16230,...,2.0,2.0,1.0,15.0,8.0,4,1,1,1.0,"16,9307528055373"
1,2025,2,2025,1,15,151,1,15101,15000012,16230,...,2.0,1.0,1.0,9.0,9.0,3,1,1,1.0,"19,8570154818132"
2,2025,2,2025,1,15,151,1,15101,15000012,16230,...,NaN,NaN,NaN,NaN,12.0,0,9,17,3.0,"17,2612656164794"
3,2025,2,2025,1,15,151,1,15101,15000012,16230,...,NaN,NaN,NaN,NaN,12.0,0,9,19,3.0,"15,4560029879504"
4,2025,2,2025,1,15,151,1,15101,15000012,16230,...,NaN,NaN,NaN,NaN,11.0,0,9,17,3.0,"14,0544783242967"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390898,2025,11,2025,12,13,131,1,13111,13100412,45280,...,NaN,NaN,NaN,NaN,2.0,0,9,15,3.0,"111,598978071473"
390899,2025,11,2025,12,15,151,1,15101,15000012,10173,...,NaN,NaN,NaN,NaN,11.0,0,9,17,3.0,"18,1443766300941"
390900,2025,11,2025,12,13,131,1,13122,13100312,12299,...,NaN,NaN,NaN,NaN,3.0,0,8,27,3.0,"135,014640338557"
390901,2025,11,2025,12,10,103,1,10301,10200112,22873,...,NaN,NaN,NaN,NaN,1.0,0,9,15,3.0,"28,7714440689824"


Forma: (390903, 222)


# ¿Qué representa cada registro en la data original?

Cada fila es **una persona entrevistada en un mes de encuesta**. Una misma persona puede aparecer una o varias veces en el año.

Los identificadores relevantes son:
- `idrph`: identificador único de la persona.
- `id_identificacion`: identificador del hogar.
- `mes_central`: trimestre móvil al que pertenece la entrevista (2, 5, 8 u 11).
- `mes_encuesta`: mes en que se realizó la entrevista (1 a 12).

In [ ]:
print('Registros (filas):', df.shape[0])
print('Columnas:', df.shape[1])
print()
print("Columnas del DataFrame mostradas de 12 en 12:")
# columnas de df mostradas de 12 en 12
for i in range(0, len(df.columns), 12):
    print(df.columns.tolist()[i:i+12])
print()
print('Personas distintas (idrph):', df['idrph'].nunique())
print('Hogares distintos (id_identificacion):', df['id_identificacion'].nunique())

Registros (filas): 390903
Columnas: 222

Columnas del DataFrame mostradas de 12 en 12:
['ano_trimestre', 'mes_central', 'ano_encuesta', 'mes_encuesta', 'region', 'provincia', 'tipo', 'r_p_c', 'estrato', 'conglomerado', 'id_identificacion', 'hogar']
['idrph', 'nro_linea', 'proveedor', 'parentesco', 'sexo', 'edad', 'est_conyugal', 'mig1', 'mig2_cod', 'mig3_cod', 'mig4', 'mig5_cod']
['mig6_cod', 'nacionalidad', 'orig1', 'orig2', 'orig3', 'curso', 'nivel', 'termino_nivel', 'estudia_actual', 'edu1', 'edu2', 'edu3']
['edu4', 'a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a6_otro', 'a7', 'a8', 'b1', 'b2']
['b3', 'b4', 'b5', 'b6', 'b7a_1', 'b7a_2', 'b7a_3', 'b7b_1', 'b7b_2', 'b7b_3', 'b7b_4', 'b8']
['b9', 'b10', 'b12', 'b13_rev4cl_caenes', 'b14_rev4cl_caenes', 'b15_1', 'b15_2', 'b16', 'b16_otro', 'b17_mes', 'b17_ano', 'b18_region']
['b18_codigo', 'b18_varias', 'b19', 'dependencia_segunda', 'i1', 'i2', 'i3', 'i3_v', 'i4', 'i5', 'i6', 'i7']
['f7a', 'f7b', 'f7c', 'f7d', 'f7e', 'de_1a', 'de_1b', 'de_2a', 'd

# 0. Determinación de subconjunto de la data original

## 0.1 Decisión y justificación

___

* <span style="color: darkred;">Considerando que la data original (390.903 filas × 222 columnas) contiene datos de personas que son encuestadas 1, 2, 3 o 4 veces durante el año, se toma la decisión de hacer una selección intencionada del subconjunto que considera los casos de sólo las 29.690 personas que han recibido exactamente las 4 visitas. Esto equivale a **118.760 filas**</span>

* **La reducción ha sido solo de filas.** El archivo del panel conserva las 222 columnas.

* **Los siguientes pasos son los que se aplican al archivo de la descarga original (ano-2025.csv) para obtener la versión reducida (dataframe `df_panel` exportable como archivo `ene2025_panel_4visitas.csv`).**

En próximas semanas o desarrollos es posible que la lectura sea directa desde este "nuevo" archivo para completar los pasos y transformaciones que se requieren en el curso.**

___

## 0.2 Pasos a realizar

| Paso | Acción | Filas antes | Filas después | Comentario |
|---|---|---|---|---|
| 1 | Carga de `ano-2025.csv` con separador `;` y codificación UTF-8 | 390.903 | 390.903 | Punto de partida |
| 2 | Revisión de duplicados y de identificadores | 390.903 | 390.903 | No hay filas duplicadas. Hay 177.402 personas (`idrph`) y unos 62.900 hogares (`id_identificacion`) |
| 3 | Conversión de `fact_anual` de texto con coma decimal a número | 390.903 | 390.903 | Corrige el tipo de dato para ponderar. No elimina registros |
| 4 | Conteo de registros por persona | 390.903 | 390.903 | Diagnóstico: 61.740 personas aparecen 1 vez, 47.513 dos veces, 38.459 tres veces y 29.690 cuatro veces |
| 5 | Selección de personas con exactamente 4 registros | 390.903 | **118.760** | Preguntas de panel: seguimiento individual y transiciones |
| 6 | Validación del panel | 118.760 | 118.760 | Cada persona tiene una visita por trimestre móvil (`mes_central` 2, 5, 8 y 11), con visitas separadas 3 meses |
| 7 | Comparación con el resto de la población | 118.760 | 118.760 | Diagnóstico de representatividad: el panel es algo mayor y tiene menos desocupación |
| 8 | Guardado del dataframe final | 118.760 | 118.760 | dataframe `df_panel` exportable como archivo `ene2025_panel_4visitas.csv` |

## 0.3. ¿Cuántas veces aparece cada persona?

Antes de elegir un subconjunto, revisamos cuántos registros tiene cada persona.

In [ ]:
registros_por_persona = df.groupby('idrph').size().rename('n_registros')

resumen = (registros_por_persona.value_counts()
           .sort_index()
           .rename('n_personas')
           .to_frame()
           .reset_index())
resumen['%_personas'] = (resumen['n_personas'] / resumen['n_personas'].sum() * 100).round(1)
resumen

,n_registros,n_personas,%_personas
0,1,61740,34.8
1,2,47513,26.8
2,3,38459,21.7
3,4,29690,16.7


## 0.4. Decisión de análisis: Panel de estudio.

* **Hay personas con 1, 2, 3 o 4 registros. Las personas con 4 registros son las únicas que aparecen en todas las mediciones del año.**
* **Se creará un panel que conserva todos esos registros**

Esto deja un grupo que tiene datos en las cuatro mediciones, sin importar el mes o el trimestre.

In [ ]:
ids_4_visitas = registros_por_persona[registros_por_persona == 4].index

panel = df[df['idrph'].isin(ids_4_visitas)].copy()

print('Personas en el panel:', panel['idrph'].nunique())
print('Registros en el panel:', len(panel))
print('Porcentaje de registros del total:', round(len(panel) / len(df) * 100, 1), '%')

Personas en el panel: 29690
Registros en el panel: 118760
Porcentaje de registros del total: 30.4 %


## 0.5. Validación de que el panel es balanceado

Verificamos que cada persona del panel cumpla dos condiciones:

1. Tiene **un registro en cada uno de los cuatro trimestres móviles** (`mes_central` = 2, 5, 8 y 11).
2. Sus visitas están **espaciadas 3 meses** entre sí.

In [ ]:
# Condición 1: un registro por trimestre móvil
trimestres_distintos = panel.groupby('idrph')['mes_central'].nunique()
print('Personas con 4 trimestres distintos:', (trimestres_distintos == 4).sum(),
      'de', len(trimestres_distintos))

# Condición 2: espaciamiento de 3 meses entre visitas
def espaciamientos(meses):
    m = sorted(meses)
    return tuple(b - a for a, b in zip(m, m[1:]))

espacios = panel.groupby('idrph')['mes_encuesta'].apply(espaciamientos)
print('Personas con espaciamiento (3, 3, 3):', (espacios == (3, 3, 3)).sum(),
      'de', len(espacios))

Personas con 4 trimestres distintos: 29690 de 29690
Personas con espaciamiento (3, 3, 3): 29690 de 29690


* Las dos condiciones se cumplen en todas las personas. Esto confirma que el panel es balanceado.
* Los meses de entrevista siguen uno de tres patrones, según el trimestre en que empieza la secuencia:

In [ ]:
patrones = (panel.sort_values('mes_encuesta')
            .groupby('idrph')['mes_encuesta']
            .agg(lambda s: tuple(s)))

conteo_patrones = patrones.value_counts().rename('n_personas').to_frame().reset_index()
conteo_patrones

,mes_encuesta,n_personas
0,"(1, 4, 7, 10)",10401
1,"(2, 5, 8, 11)",9812
2,"(3, 6, 9, 12)",9477


## 0.6. Qué tan parecido es este grupo al total

**El panel no es una muestra aleatoria. Quienes permanecen en la encuesta todo el año pueden ser distintos del resto. Comparamos el **primer registro** de cada persona, dentro y fuera del panel, para ver las diferencias.**

In [ ]:
primer_registro = df.sort_values(['idrph', 'mes_encuesta']).groupby('idrph').head(1).copy()
primer_registro = primer_registro[primer_registro['edad'] >= 15].copy()
primer_registro['en_panel'] = primer_registro['idrph'].isin(ids_4_visitas)

def resumen_grupo(g):
    return pd.Series({
        'personas': len(g),
        'mujeres_%': round((g['sexo'] == 2).mean() * 100, 1),
        'edad_media': round(g['edad'].mean(), 1),
        'ocupados_%': round((g['activ'] == 1).mean() * 100, 1),
        'desocupados_%': round((g['activ'] == 2).mean() * 100, 1),
        'fuera_FT_%': round((g['activ'] == 3).mean() * 100, 1),
        'ciudad_%': round((g['tipo'] == 1).mean() * 100, 1),
        'rural_%': round((g['tipo'] == 3).mean() * 100, 1),
    })

comparacion = pd.DataFrame({
    'Fuera del panel': resumen_grupo(primer_registro[~primer_registro['en_panel']]),
    'Panel 4 visitas': resumen_grupo(primer_registro[primer_registro['en_panel']]),
})
comparacion

,Fuera del panel,Panel 4 visitas
personas,123037.0,24970.0
mujeres_%,52.9,54.3
edad_media,46.2,48.6
ocupados_%,53.1,51.0
desocupados_%,5.0,4.3
fuera_FT_%,41.9,44.7
ciudad_%,79.0,77.5
rural_%,21.0,22.5


Las diferencias son moderadas. El panel tiene una edad media algo mayor y menos desocupación. Esto debe tenerse en cuenta al interpretar los resultados.

## 0.7. Dataframe final

Revisamos que el resultado tenga la forma esperada y lo guardamos para los análisis posteriores.

In [ ]:
df_panel = panel.copy()

assert len(df_panel) == 118760, 'El número de registros no coincide'
assert df_panel['idrph'].nunique() == 29690, 'El número de personas no coincide'
assert df_panel.groupby('idrph').size().eq(4).all(), 'Hay personas con distinto número de visitas'

print('Registros:', df_panel.shape[0])
print('Personas:', df_panel['idrph'].nunique())
print('Columnas:', df_panel.shape[1])

nombre_panel_salida = 'ene2025_panel_4visitas.csv'

if not os.path.exists(RUTA + nombre_panel_salida):
    df_panel.to_csv(RUTA + nombre_panel_salida, sep=';', index=False, encoding='utf-8')
    print(f'Archivo guardado: {nombre_panel_salida}')
else:
    print(f'Archivo no guardado: {nombre_panel_salida} ya existe')

Registros: 118760
Personas: 29690
Columnas: 222
Archivo no guardado: ene2025_panel_4visitas.csv ya existe


**Simulación de la lectura directa del archivo de panel**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os

RUTA = '/content/drive/Othercomputers/Notebook HP/Ing Informática UCSC/Trimestre 3/1 Ciencia de datos/Evaluaciones/Semana 1/Lab 1/'
SEMILLA = 20262
np.random.seed(SEMILLA)

In [ ]:
nombre_panel_salida = 'ene2025_panel_4visitas.csv'

try:
    df = pd.read_csv(RUTA + nombre_panel_salida, sep=';', encoding='utf-8',
                    low_memory=False, dtype={'fact_anual': str})
    display(df)
except Exception as e:
    print(type(e).__name__)
    print(e)

print('Forma:', df.shape)

,ano_trimestre,mes_central,ano_encuesta,mes_encuesta,region,provincia,tipo,r_p_c,estrato,conglomerado,...,ina_cal_edu,ocup_form,sector,r_p_rev4cl_caenes,tramo_edad,categoria_ocupacion,cae_general,cae_especifico,activ,fact_anual
0,2025,2,2025,1,13,131,1,13110,13100312,20043,...,NaN,NaN,NaN,NaN,12.0,0,9,19,3.0,93.9248633043816
1,2025,2,2025,1,13,131,1,13110,13100313,36033,...,2.0,1.0,1.0,16.0,12.0,4,1,1,1.0,102.101514163956
2,2025,2,2025,1,13,131,1,13110,13100313,36033,...,NaN,NaN,NaN,NaN,10.0,0,9,13,3.0,97.3545793226457
3,2025,2,2025,1,3,33,1,3301,3200110,14530,...,NaN,NaN,NaN,NaN,12.0,0,9,21,3.0,15.7439924130857
4,2025,2,2025,1,8,81,1,8101,8100312,13536,...,NaN,NaN,NaN,NaN,11.0,0,8,16,3.0,33.6905139627863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118755,2025,11,2025,12,6,61,3,6111,6100120,28801,...,NaN,NaN,NaN,NaN,12.0,0,9,19,3.0,23.0665763227126
118756,2025,11,2025,12,6,61,3,6111,6100120,28801,...,NaN,NaN,NaN,NaN,12.0,0,9,19,3.0,22.3245738963905
118757,2025,11,2025,12,8,81,1,8112,8100312,31712,...,NaN,NaN,NaN,NaN,7.0,0,9,13,3.0,37.2813912428056
118758,2025,11,2025,12,7,71,1,7109,7100310,31106,...,NaN,NaN,NaN,NaN,11.0,0,9,21,3.0,25.8120651873786


Forma: (118760, 222)


**Población de referencia:** Estos registros corresponden a las personas que estuvieron en la encuesta en los cuatro trimestres. No representan a toda la población original

# Selección arbitraria de columnas (Primer paso de la bitácora)

Es momento de reducir la cantidad de columnas y tomar una decisión de cuáles dejar. Esto porque es demasiado para manejar. Estimo mantener máximo 30 columnas pensando en análisis posteriores interesantes.

Propongo **30 columnas**, agrupadas por la función que cumplen en el análisis. Todas existen en el panel. Algunas no sirven como predictores de la desocupación, pero sí para estudiar transiciones o describir el empleo.

## Lista selecionada

**1. Diseño y claves técnicas (6)**
- `idrph`: identificador de persona. Solo para construir transiciones; se elimina antes de compartir.
- `mes_central`, `mes_encuesta`: trimestre y mes de la visita.
- `fact_anual`: ponderador.
- `conglomerado`, `estrato`: diseño muestral.

**2. Variable objetivo (1)**
- `activ`: condición laboral (ocupado, desocupado, inactivo).

**3. Territorio (2)**
- `region`, `tipo`.

**4. Sociodemográficas (8)**
- `sexo`, `edad`, `nacionalidad`, `orig1`, `est_conyugal`, `estudia_actual`, `parentesco`, `proveedor`.

**5. Educación (3)**
- `nivel`, `termino_nivel`, `cine11_1d`.

**6. Estado de empleo y descriptivas (8)**
- `categoria_ocupacion`: asalariado, cuenta propia, empleador, etc. Es clave para tu análisis de dependencia.
- `cae_general`: rama de actividad.
- `ocup_form`: formalidad.
- `ina_cal_edu`: inadecuación por calificación.
- `habituales`, `meses_ocup_actual`: jornada y antigüedad.
- `no_prevision`: asalariado sin cotizaciones.
- `plataformas_digitales`: trabajo a través de apps.

**7. Inactividad (2)**
- `deseo_trabajar`, `ftp`: distinguen inactivos con deseo o disponibilidad de trabajar.

## Cómo usarlas sin trampas

Las 30 no sirven igual para todo:

- **Predictores de la desocupación (15):** `sexo`, `edad`, `nacionalidad`, `orig1`, `est_conyugal`, `estudia_actual`, `parentesco`, `proveedor`, `nivel`, `termino_nivel`, `cine11_1d`, `region`, `tipo`, y los pesos y diseño. Son las únicas que cumplen tu pregunta de factores sin usar la propia condición.
- **Estado y transiciones (8):** `categoria_ocupacion`, `cae_general`, `ocup_form`, `ina_cal_edu`, `habituales`, `meses_ocup_actual`, `no_prevision`, `plataformas_digitales`. Describen el empleo de cada visita, pero no deben usarse para predecir `activ` en la misma visita, porque la definen.
- **Inactividad (2):** `deseo_trabajar`, `ftp`. Útiles para caracterizar el paso de inactivo a desocupado, no como predictores de la condición actual.

Las columnas de estado tienen 50 % de vacíos, porque solo las responden ocupados. Eso es correcto, pero los cálculos deben hacerse dentro de cada grupo.

## Qué dejé fuera y por qué

- **Detalle de búsqueda (`e3_*`, `e5_*`, `e6_*`, `e7`, `e9`, `e10` a `e12`):** muy vacías y definen la desocupación. Si necesitas el detalle, usa un análisis separado.
- **Jornada y turnos (`c*`, `turno_*`, `adicionales_*`, `c2_2_*`):** casi todas vacías y específicas de ocupados.
- **Contratos y derechos (`b7*`, `b8` a `b10`):** salvo `no_prevision`, casi redundantes con ella.
- **Empresa y pago (`i*`, `re*`, `b11_proxy`):** muy vacías, y `i3` es texto libre y sensible.
- **Independientes (`de_*`, `do_*`, `dep_*`, `cd*`):** solo para un subconjunto pequeño.
- **Derivadas de la condición (`tpi`, `obe`, `ft`, `fta`, `inhab`, `idisp`):** definen la condición laboral, así que generan fuga de información.
- **Textos y "otro" (`*_otro`, `*_especifique`, `*_cod`):** se excluyen.
- **Redundantes:** `edad` ya cubre `tramo_edad`, y `edu1` a `edu4` repiten `nivel`.

**Detalle 30 columnas**

| Campo | Definición (catálogo) | Valores posibles (catálogo) |
|---|---|---|
| `idrph` | Identificador único de personas | Sin códigos definidos |
| `mes_central` | Trimestre móvil | 1: Diciembre-febrero; 2: Enero-marzo; 3: Febrero-abril; 4: Marzo-mayo; 5: Abril-junio; 6: Mayo-julio; 7: Junio-agosto; 8: Julio-septiembre; 9: Agosto-octubre; 10: Septiembre-noviembre; 11: Octubre-diciembre; 12: Noviembre-enero |
| `mes_encuesta` | Mes de recolección de datos | 1: Enero; 2: Febrero; 3: Marzo; 4: Abril; 5: Mayo; 6: Junio; 7: Julio; 8: Agosto; 9: Septiembre; 10: Octubre; 11: Noviembre; 12: Diciembre |
| `fact_anual` | Factor de expansión anual | Numérico |
| `conglomerado` | Conglomerado (unidad primaria de muestreo) | Sin códigos definidos |
| `estrato` | Estrato muestral | Sin códigos definidos |
| `activ` | Condición de actividad | 1: Ocupado/a; 2: Desocupado/a; 3: Fuera de la fuerza de trabajo |
| `region` | Región | Ver Anexo 1 (detalle de regiones) |
| `tipo` | Tipo de estrato | 1: Ciudad (CD); 2: Resto de Área Urbana (RAU); 3: Rural |
| `sexo` | Sexo | 1: Hombre; 2: Mujer |
| `edad` | Edad | 0-120 |
| `nacionalidad` | Nacionalidad | Ver Anexo 2 (detalle de países); 898: Zona no especificada; 999: Sin información |
| `orig1` | ¿Se considera perteneciente a algún pueblo indígena u originario? | 1: Sí; 2: No; 88: No sabe; 99: No responde |
| `est_conyugal` | Estado civil o conyugal | 0: No aplica; 1: Casado/a; 2: Conviviente; 3: Soltero/a; 4: Viudo/a; 5: Separado/a de hecho o anulado/a; 6: Divorciado/a; 88: No sabe; 99: No responde |
| `estudia_actual` | ¿Se encuentra estudiando actualmente? | 1: Sí; 2: No; 88: No sabe; 99: No responde |
| `parentesco` | Parentesco con proveedor/a principal del hogar | 1: Persona proveedora principal; 2: Cónyuge; 3: Conviviente; 4: Hijo/a o hijastro/a; 5: Yerno o nuera; 6: Nieto/a; 7: Hermano/a o cuñado/a; 8: Padres o suegros; 9: Otro pariente; 10: No pariente; 11: Servicio doméstico; 88: No sabe; 99: No responde |
| `proveedor` | Proveedor/a principal del hogar | 0: No corresponde; 1: Proveedor/a principal |
| `nivel` | Nivel educacional más alto aprobado | 0: Nunca estudió; 1: Sala cuna; 2: Kínder; 3: Básica o primaria; 4: Media común; 5: Media técnico profesional; 6: Humanidades; 7: Centro de formación técnica; 8: Instituto profesional; 9: Universitario; 10: Post títulos; 11: Magíster; 12: Doctorado; 13: Educación (texto truncado en la extracción, revisar el catálogo) |
| `termino_nivel` | Término de nivel educacional | 1: Sí; 2: No; 88: No sabe; 99: No responde |
| `cine11_1d` | CINE 2011 (nivel, 1 dígito) | 0: Menos que primaria; 1: Primaria; 2: Secundaria; 3: Terciaria ciclo corto; 4: Grado terciario; 5: Maestría; 6: Doctorado; 9: Sin clasificación |
| `categoria_ocupacion` | Categoría en la ocupación | 0: No corresponde; 1: Empleador; 2: Cuenta propia; 3: Asalariado privado; 4: Asalariado público; 5: Servicio doméstico afuera; 6: Servicio doméstico adentro; 7: Familiar no remunerado |
| `cae_general` | Código sumario de empleo general | 0: Menor de 15 años; 1: Ocupado tradicional; 2: Ocupado no tradicional; 3: Ocupado ausente; 4: Cesante; 5: Buscó trabajo por 1° vez; 6: Iniciador/a; 7: Inactivo que buscó; 8: Inactivo disponible; 9: Inactivo no buscó ni disponible |
| `ocup_form` | Ocupado/a según formalidad de su ocupación | 1: Ocupado formal; 2: Ocupado informal |
| `ina_cal_edu` | Inadecuación por calificación según nivel educativo | 1: Subeducación; 2: Sin inadecuación; 3: Sobreeducación; 999: Sin información |
| `habituales` | Horas habituales de trabajo | 1-168; 888: No sabe; 999: No responde |
| `meses_ocup_actual` | Duración de la ocupación declarada (en meses) | Meses (numérico) |
| `no_prevision` | Persona ocupada dependiente cuyo empleador no paga cotizaciones | 0: No corresponde; 1: Empleador no paga cotizaciones |
| `plataformas_digitales` | ¿Su trabajo lo realiza a través de app o plataforma web? | 1: Sí; 2: No; 77: No aplica; 88: No sabe; 99: No responde |
| `deseo_trabajar` | A pesar de que no buscó trabajo, ¿desea trabajar? | 1: Sí; 2: No; 88: No sabe; 99: No responde |
| `ftp` | Fuerza de trabajo potencial | 0: No corresponde; 1: Fuerza de trabajo potencial |

Notas:

- Donde el catálogo dice "Sin códigos definidos" o "-", la columna no tiene una lista de valores documentada.
- Las columnas `region` y `nacionalidad` remiten a anexos del libro de códigos (Anexo 1 y Anexo 2).

In [ ]:
# Paso 1 para bitácora

cols_analisis = [
    'idrph', 'mes_central', 'mes_encuesta', 'fact_anual', 'conglomerado', 'estrato',
    'activ', 'region', 'tipo',
    'sexo', 'edad', 'nacionalidad', 'orig1', 'est_conyugal', 'estudia_actual', 'parentesco', 'proveedor',
    'nivel', 'termino_nivel', 'cine11_1d',
    'categoria_ocupacion', 'cae_general', 'ocup_form', 'ina_cal_edu',
    'habituales', 'meses_ocup_actual', 'no_prevision', 'plataformas_digitales',
    'deseo_trabajar', 'ftp',
]
df = df[cols_analisis].copy()

In [ ]:
print('Registros (filas):', df.shape[0])
print('Columnas:', df.shape[1])
print()
print("Columnas del DataFrame mostradas de 12 en 12:")
# columnas de df mostradas de 12 en 12
for i in range(0, len(df.columns), 12):
    print(df.columns.tolist()[i:i+12])
print()
print('Personas distintas (idrph):', df['idrph'].nunique())

Registros (filas): 118760
Columnas: 30

Columnas del DataFrame mostradas de 12 en 12:
['idrph', 'mes_central', 'mes_encuesta', 'fact_anual', 'conglomerado', 'estrato', 'activ', 'region', 'tipo', 'sexo', 'edad', 'nacionalidad']
['orig1', 'est_conyugal', 'estudia_actual', 'parentesco', 'proveedor', 'nivel', 'termino_nivel', 'cine11_1d', 'categoria_ocupacion', 'cae_general', 'ocup_form', 'ina_cal_edu']
['habituales', 'meses_ocup_actual', 'no_prevision', 'plataformas_digitales', 'deseo_trabajar', 'ftp']

Personas distintas (idrph): 29690


In [ ]:
df.dtypes

,0
idrph,int64
mes_central,int64
mes_encuesta,int64
fact_anual,object
conglomerado,int64
estrato,int64
activ,float64
region,int64
tipo,int64
sexo,int64


In [ ]:
# print de cada tipo de dato y lista de las columnas por cada tipo

for tipo, columnas in df.dtypes.groupby(df.dtypes).groups.items():
    print(f'Tipo: {tipo}')
    print(f'Columnas: {list(columnas)}\n')

Tipo: int64
Columnas: ['idrph', 'mes_central', 'mes_encuesta', 'conglomerado', 'estrato', 'region', 'tipo', 'sexo', 'edad', 'nacionalidad', 'est_conyugal', 'estudia_actual', 'parentesco', 'proveedor', 'nivel', 'termino_nivel', 'cine11_1d', 'categoria_ocupacion', 'cae_general']

Tipo: float64
Columnas: ['activ', 'orig1', 'ocup_form', 'ina_cal_edu', 'habituales', 'no_prevision', 'plataformas_digitales', 'deseo_trabajar', 'ftp']

Tipo: object
Columnas: ['fact_anual', 'meses_ocup_actual']



## Detalles adicionales sobre los vacíos y su pertinencia.

Los vacíos se concentran en pocos bloques temáticos. Cada bloque tiene una lógica de filtro: solo responde un subgrupo, según su condición laboral.

### 1. Campos de texto libre o "otro, especifique"

Son los vacíos más altos (99 % o más). Solo se llenan cuando la persona elige la opción "otro" y escribe la razón.

- `a6_otro`, `c9_otro`, `b16_otro`, `e9_otro`, `e12_otro`, `e23_otro`, `e24_otro`: especifican la opción "otra razón" de las preguntas a6, c9, b16, e9, e12, e23 y e24.
- `orig3`: especifica el pueblo originario cuando se elige "otro" en `orig2`.
- `sda_pd_especifique` y `pd_especifique`: nombre de la aplicación o plataforma digital.
- `mig3_cod`, `mig6_cod`: códigos del país de migración.

**Recomendación:** excluir del análisis.

### 2. Ausencia del trabajo en la semana de referencia (a4 a a8)

Estas preguntas solo las responden los ocupados que no trabajaron la semana de referencia, por ejemplo por vacaciones o licencia. Por eso tienen cerca de 99 % de vacíos.

- `a4`: si el trabajo era para una empresa de alguien del hogar.
- `a6`: razón por la que no trabajó la semana pasada.
- `a7`: si siguió recibiendo sueldo durante la ausencia.
- `a8`: en cuánto tiempo volverá a trabajar.

### 3. Jornada y horas (c-series, turno_*, c2_2_*, adicionales_*)

- `c2_1_1` a `c2_1_3`: horas diarias, días y total semanal habitual de la actividad principal.
- `c2_2_1` a `c2_2_3`: lo mismo para la actividad secundaria. Casi nadie tiene segunda actividad, por eso tiene 98 % de vacíos.
- `c4`, `c5`, `c6`, `c7`, `c8`: preguntas sobre horas extra y horas menos de las habituales la semana pasada.
- `c9`, `c10`, `c11`, `c12`: razones por las que trabajó otras horas, y disponibilidad para trabajar más.
- `adicionales_h`, `adicionales_d`, `adicionales_t`: horas, días y total que podría trabajar adicionalmente.
- `turno`, `turno_h`, `turno_d`, `turno_t`, `turno_de` y variantes `turno_cont_*`: si trabaja por turnos y cuántas horas y días, habituales o contratados.

Estos bloques solo los responden ocupados que cumplen condiciones específicas.

### 4. Búsqueda de empleo (e3 a e12)

Son las preguntas que definen al desocupado.

- `e3_1` a `e3_12` y `e3_total`: gestiones de búsqueda realizadas (currículum, avisos, contactos, etc.). Los desocupados las responden casi completas, pero los ocupados casi no.
- `e4`: motivo principal por el que busca otro trabajo.
- `e5`, `e5_dia`, `e5_mes`, `e5_ano`: cuándo fue la última vez que buscó trabajo.
- `e6_mes`, `e6_ano`: desde cuándo busca trabajo.
- `e7`: tipo de jornada que busca.
- `e9`: razón por la que no buscó en las últimas cuatro semanas.
- `e10`: si hizo gestiones para iniciar un negocio.
- `e11`: si estaría disponible para empezar si encontrara trabajo.
- `e12`: motivo por el que no estaría disponible.

### 5. Historial de término de empleo (e21 a e24)

Solo responden quienes dejaron un trabajo anterior.

- `e21_mes`, `e21_ano`, `e21_tramo`: hasta cuándo trabajó en esa ocupación.
- `e22`: razón por la que ya no tiene ese empleo.
- `e23`: motivo del despido.
- `e24`: motivo de renuncia.

### 6. Trabajo independiente y dependencia (de_*, do_*, dep_*, cd, cdi, n_dependencias)

Describen la autonomía del trabajador independiente.

- `de_1a`, `de_1b`: si requiere local o predio, y quién es el propietario.
- `de_2a`, `de_2b`: si usa maquinaria, y quién es el propietario.
- `de_3a`, `de_3b`: si usa vehículo de transporte, y quién es el propietario.
- `do_1` a `do_4`: quién decide sobre horario, rutas, tipo de producto y precio.
- `dep_economica`, `dep_organizativa`, `dep_ambas`: dependencia económica u organizativa de los independientes.
- `cd`, `cdi`, `cdd`: si es contratista dependiente.
- `n_dependencias`: número de ámbitos de dependencia.

Solo los ocupados independientes tienen estas preguntas, por eso tienen cerca de 88 % de vacíos.

### 7. Contrato y previsión (b7a_*, b7b_*, b8, b9, b10, no_prevision, no_salario)

Describen derechos laborales de los asalariados.

- `b7a_1`, `b7a_2`, `b7a_3`: si el empleador cotiza en pensiones, salud y seguro de desempleo.
- `b7b_1` a `b7b_4`: si tiene derecho a vacaciones, días por enfermedad, permiso por maternidad o paternidad y guardería.
- `b8`: si tiene contrato escrito.
- `b9`: duración del contrato.
- `b10`: tipo de contrato o acuerdo.
- `no_prevision`: asalariado cuyo empleador no paga cotizaciones.
- `no_salario`: ocupado que recibe un pago diferente a sueldo.

Estas preguntas no aplican a independientes, por eso tienen cerca de 63 % de vacíos.

### 8. Empresa y formalidad (i1 a i7, i3, i3_v, i4 a i7)

- `i1`: si la empresa está registrada en el SII o tiene inicio de actividades.
- `i2`: si cuenta con contador.
- `i3`: nombre de la empresa. Es texto libre y dato sensible, así que conviene excluirlo.
- `i3_v`: verificación de la respuesta con el Directorio de Empresas del INE.
- `i4`, `i5`: registro en el SII de la empresa o del negocio.
- `i6`, `i7`: si la actividad por cuenta propia separa gastos del hogar, y contabilidad.

### 9. Forma de pago (re1 a re6, b11_proxy)

Indican cómo se pagó el trabajo: sueldo, comisión, propina, pieza vendida, cobro por servicio o utilidades. Son respuestas de tipo sí o no, no montos de ingreso. El catálogo no trae una variable de ingreso monetario en 2025, así que si el análisis necesita ingresos, debe buscarse en otra fuente.

### 10. Plataformas digitales (pd_*, sda_pd*, plataformas_digitales)

Indican si el trabajo se realiza a través de una app o plataforma web, y si está sujeto a la Ley 21.431. `sda_pd` se refiere al segundo trabajo.

### 11. Variables derivadas (ocup_form, sector, tpi, ina_cal_edu, ciso*, obe, ft, ftp, fta, inhab, idisp)

Se calculan a partir de la condición laboral y solo existen para ciertos grupos.

- `ocup_form`, `sector`: formalidad y sector institucional de la ocupación.
- `tpi`: ocupado a tiempo parcial involuntario.
- `ina_cal_edu`: inadecuación por calificación según nivel educativo.
- `ciso1`, `ciso2`, `ciso3_a`, `ciso3_b`: situación en la ocupación, según la clasificación CISO-18.
- `obe`: ocupado que buscó empleo.
- `ft`, `ftp`, `fta`: fuerza de trabajo, fuerza de trabajo potencial y ampliada.
- `inhab`: inactivo habitual.
- `idisp`: iniciador disponible.

Un dato útil: estas variables tienen alrededor de 50 % de vacíos entre personas de 15 años o más, y los ocupados son también alrededor del 50 %. Es decir, están vacías para los no ocupados, lo que confirma que solo describen a quienes trabajan.

## Resumen

Los vacíos responden a tres causas:

1. **Filtro por condición laboral:** bloques para ocupados, desocupados o independientes.
2. **Filtro por respuesta anterior:** preguntas que se activan solo si la respuesta previa tenía un valor concreto.
3. **Texto libre o "otro":** campos que casi nadie completa.

## Parte 1 · Diagnóstico de las cinco dimensiones

Antes de limpiar nada, hay que saber qué tienes. Completa la hoja **«Resumen de calidad»** de la bitácora con los resultados de esta parte.

### 1.1 Completitud

In [ ]:
faltantes = df.isna().sum().to_frame('faltantes')
faltantes['pct'] = (faltantes['faltantes'] / len(df) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,pct
no_prevision,81328,68.48
deseo_trabajar,74828,63.01
ftp,72810,61.31
plataformas_digitales,68654,57.81
ina_cal_edu,68654,57.81
ocup_form,68654,57.81
meses_ocup_actual,68654,57.81
habituales,68654,57.81
activ,18321,15.43
orig1,1,0.00


In [ ]:
lista_columnas_sin_vacios = faltantes[faltantes["pct"] == 0].index.tolist()
print(f'''Columnas sin vacíos ({len(lista_columnas_sin_vacios)}):
{chr(10).join([f'- {col}' for col in lista_columnas_sin_vacios])}
''')

Columnas sin vacíos (21): 
- idrph
- mes_central
- mes_encuesta
- fact_anual
- conglomerado
- estrato
- region
- tipo
- sexo
- edad
- nacionalidad
- orig1
- est_conyugal
- estudia_actual
- parentesco
- proveedor
- nivel
- termino_nivel
- cine11_1d
- categoria_ocupacion
- cae_general



In [ ]:
lista_columnas_vacios_20pct = faltantes[(faltantes['pct'] > 0) & (faltantes['pct'] <= 20)].index.tolist()
print(f'''Columnas con vacíos entre 0 y 20% ({len(lista_columnas_vacios_20pct)}):
{chr(10).join([f'- {col}' for col in lista_columnas_vacios_20pct])}
''')

Columnas con vacíos entre 0 y 20% (1): 
- activ



In [ ]:
lista_columnas_vacios_40pct = faltantes[(faltantes['pct'] > 20) & (faltantes['pct'] <= 40)].index.tolist()
print(f'''Columnas con vacíos entre 20 y 40% ({len(lista_columnas_vacios_40pct)}):
{chr(10).join([f'- {col}' for col in lista_columnas_vacios_40pct])}
''')

Columnas con vacíos entre 20 y 40% (0): 




In [ ]:
lista_columnas_vacios_60pct = faltantes[(faltantes['pct'] > 40) & (faltantes['pct'] <= 60)].index.tolist()
print(f'''Columnas con vacíos entre 40 y 60% ({len(lista_columnas_vacios_60pct)}):
{chr(10).join([f'- {col}' for col in lista_columnas_vacios_60pct])}
''')

Columnas con vacíos entre 40 y 60% (5): 
- ocup_form
- ina_cal_edu
- habituales
- meses_ocup_actual
- plataformas_digitales



In [ ]:
lista_columnas_vacios_80pct = faltantes[(faltantes['pct'] > 60) & (faltantes['pct'] <= 80)].index.tolist()
print(f'''Columnas con vacíos entre 60 y 80% ({len(lista_columnas_vacios_80pct)}):
{chr(10).join([f'- {col}' for col in lista_columnas_vacios_80pct])}
''')

Columnas con vacíos entre 60 y 80% (3): 
- no_prevision
- deseo_trabajar
- ftp



In [ ]:
lista_columnas_vacios_100pct = faltantes[(faltantes['pct'] > 80) & (faltantes['pct'] <= 100)].index.tolist()
print(f'''Columnas con vacíos entre 80 y 100% ({len(lista_columnas_vacios_100pct)}):
{chr(10).join([f'- {col}' for col in lista_columnas_vacios_100pct])}
''')

Columnas con vacíos entre 80 y 100% (0): 




Con las 30 columnas, en el panel solo aparecen **6 patrones distintos** de vacíos:

### Qué queda

Entre personas de 15 años o más (100.439 registros), 21 de las 30 columnas no tienen vacíos. Las 9 que sí los tienen se agrupan así:

| Filtro | Columnas | Quién las responde | Vacío fuera de ese grupo |
|---|---|---|---|
| Ocupados | `ocup_form`, `ina_cal_edu`, `habituales`, `meses_ocup_actual`, `plataformas_digitales` | Solo ocupados | 100 % en desocupados e inactivos |
| Asalariados | `no_prevision` | Ocupados dependientes | 100 % en desocupados e inactivos, y 25 % en ocupados (los independientes) |
| No ocupados | `deseo_trabajar`, `ftp` | Solo inactivos | 100 % en ocupados y desocupados |

Las cinco columnas del primer filtro tienen correlación phi = 1 entre sí, es decir, vacíos idénticos. `deseo_trabajar` y `ftp` tienen phi = 0,96. Entre el filtro de ocupados y el de no ocupados la correlación es negativa, como se espera de grupos excluyentes.

`orig1` tiene un solo vacío de 100.439 casos, que no afecta el análisis.

### Los seis patrones

Cada patrón corresponde a un grupo de personas:

| Patrón de vacíos | Registros | Grupo |
|---|---|---|
| Solo `no_prevision` vacío en ocupados, y los filtros de no ocupados respondidos | 12.674 | Ocupados independientes |
| Ocupados sin vacíos en su filtro | 37.432 | Ocupados dependientes (asalariados) |
| Todo el filtro de ocupados vacío y `deseo_trabajar`, `ftp` respondidos | 43.931 | Inactivos que respondieron |
| Todo el filtro de ocupados vacío y `deseo_trabajar`, `ftp` vacíos | 4.383 | Desocupados |
| Todo el filtro de ocupados vacío, `deseo_trabajar` vacío, `ftp` respondido | 2.018 | Inactivos sin respuesta en `deseo_trabajar` |
| Un solo caso con `orig1` vacío | 1 | Caso aislado |

**1. Algunas columnas esconden sus vacíos.** `categoria_ocupacion` y `cae_general` tienen 0 % de vacíos porque el catálogo las codifica con 0 para "no corresponde". Por eso no aparecen en el análisis de vacíos, aunque para no ocupados su valor sea 0. Al calcular porcentajes sobre ocupados, filtra primero.

**2. Un código no coincide con el catálogo.** El catálogo define `ftp` con 0 = "no corresponde" y 1 = "fuerza de trabajo potencial". En el archivo, los no inactivos aparecen como vacíos, no como 0. Conviene verificarlo antes de documentarlo, porque cambia cómo se interpreta esa columna.

Los 2.018 inactivos sin respuesta en `deseo_trabajar` también merecen revisión. Pueden ser no respuestas reales o un código especial sin documentar.

### ✏️ Actividad 1.1

Elige la columna con más faltantes y averigua **si se reparten al azar o se concentran en algún grupo**. Usa `pd.crosstab` contra alguna columna categórica, como en el Ejercicio 1 del cuaderno resuelto.

Es la diferencia entre poder imputar y tener que declarar.

In [ ]:
cols = ['idrph','mes_central','mes_encuesta','fact_anual','conglomerado','estrato',
        'activ','region','tipo','sexo','edad','nacionalidad','orig1','est_conyugal',
        'estudia_actual','parentesco','proveedor','nivel','termino_nivel','cine11_1d',
        'categoria_ocupacion','cae_general','ocup_form','ina_cal_edu','habituales',
        'meses_ocup_actual','no_prevision','plataformas_digitales','deseo_trabajar','ftp']

d = df[cols].copy()
edad = pd.to_numeric(d['edad'], errors='coerce')
p = d[edad >= 15].copy()

def cruce_vacios(df, col1, col2):
    """Tabla 2x2 que cruza si dos columnas están vacías o no."""
    v1 = df[col1].isna().map({True: 'Vacío', False: 'Responde'})
    v2 = df[col2].isna().map({True: 'Vacío', False: 'Responde'})
    return pd.crosstab(v1, v2, margins=True, margins_name='Total')

In [ ]:
cruce_vacios(p, 'ocup_form', 'no_prevision')

no_prevision,Responde,Vacío,Total
ocup_form,,,
Responde,37432,12674,50106
Vacío,0,50333,50333
Total,37432,63007,100439


### 1.2 Exactitud — rangos y valores imposibles

In [ ]:
# describe de las columnas con máximo 20% de vacíos y sin vacíos
df[lista_columnas_sin_vacios + lista_columnas_vacios_20pct].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
idrph,118760.0,4.973828e+11,2.898777e+11,27776342.0,2.461819e+11,4.942541e+11,7.483368e+11,9.999455e+11
mes_central,118760.0,6.500000e+00,3.350000e+00,2.0,4.250000e+00,6.500000e+00,8.750000e+00,1.100000e+01
mes_encuesta,118760.0,6.470000e+00,3.450000e+00,1.0,3.750000e+00,6.500000e+00,9.250000e+00,1.200000e+01
conglomerado,118760.0,2.768763e+04,1.032394e+04,10004.0,1.872700e+04,2.759100e+04,3.651700e+04,4.585700e+04
estrato,118760.0,8.738657e+06,4.068964e+06,1100111.0,5.500120e+06,8.100312e+06,1.310021e+07,1.630012e+07
region,118760.0,8.550000e+00,4.100000e+00,1.0,5.000000e+00,8.000000e+00,1.300000e+01,1.600000e+01
tipo,118760.0,1.450000e+00,8.300000e-01,1.0,1.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00
sexo,118760.0,1.540000e+00,5.000000e-01,1.0,1.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00
edad,118760.0,4.251000e+01,2.330000e+01,0.0,2.200000e+01,4.300000e+01,6.200000e+01,1.040000e+02
nacionalidad,118760.0,1.632700e+02,8.686000e+01,32.0,1.520000e+02,1.520000e+02,1.520000e+02,8.620000e+02


### ✏️ Actividad 1.2

Mira los mínimos y máximos. **¿Hay valores imposibles para lo que la columna representa?** Busca en particular valores centinela: `-1`, `0`, `999`, `9999`, `-999`, o cadenas como `'S/I'`, `'NULL'`, `'sin dato'`.

Recuerda: un centinela numérico **no** se delata en los tipos de dato.

In [ ]:
# Función detecta los centinelas que aparecen en tu captura, por ejemplo `88` y `99` en campos de respuesta, `888` y `999` en horas, y `9999` en años. Aquí está el código completo:

# Candidatos a centinela. El 0 suele ser un código válido (por ejemplo "no corresponde"),
# así que se marca pero se revisa contra el catálogo antes de tratarlo como vacío.
CANDIDATOS_NUM = [-9999, -999, -1, 0, 88, 99, 888, 999, 8888, 9999]
CANDIDATOS_TXT = ['S/I', 'NULL', 'SIN DATO', 'NA', 'N/A', 'NO SABE', 'NO RESPONDE']
EXCLUIR = ['idrph', 'id_identificacion', 'hogar', 'nro_linea']  # identificadores: no se evalúan

def detectar_centinelas(df, cand_num=CANDIDATOS_NUM, cand_txt=CANDIDATOS_TXT, excluir=EXCLUIR):
    """Busca valores candidatos a centinela en cada columna.
    Devuelve un resumen por columna (ordenado por % de centinelas) y el detalle por valor."""
    n = len(df)
    resumen, detalle = [], []
    for col in df.columns:
        if col in excluir:
            continue
        s = df[col].dropna()
        if s.empty:
            continue
        num = pd.to_numeric(s, errors='coerce')
        es_numerica = num.notna().mean() > 0.95
        hallazgos = []
        if es_numerica:
            for v in cand_num:
                k = int((num == v).sum())
                if k > 0:
                    hallazgos.append((str(v), k))
        txt = s.astype(str).str.strip().str.upper()
        for v in cand_txt:
            k = int((txt == v).sum())
            if k > 0:
                hallazgos.append((v, k))
        resumen.append({
            'columna': col,
            'tipo': 'numérica' if es_numerica else 'texto',
            'min': num.min() if es_numerica else None,
            'max': num.max() if es_numerica else None,
            'candidatos_encontrados': ', '.join(f'{v} ({k})' for v, k in hallazgos) or '-',
            'pct_centinela': round(sum(k for _, k in hallazgos) / n * 100, 2),
        })
        for v, k in hallazgos:
            detalle.append({'columna': col, 'valor': v, 'casos': k,
                            'pct_del_total': round(k / n * 100, 2)})
    resumen = pd.DataFrame(resumen).sort_values('pct_centinela', ascending=False)
    detalle = pd.DataFrame(detalle).sort_values(['columna', 'casos'], ascending=[True, False])
    return resumen.reset_index(drop=True), detalle.reset_index(drop=True)

In [ ]:
# ---------- Uso ----------
resumen, detalle = detectar_centinelas(df)

# Columnas con al menos un candidato, ordenadas por % de centinelas
con_candidatos = resumen[resumen['candidatos_encontrados'] != '-']

con_candidatos.head(30)

,columna,tipo,min,max,candidatos_encontrados,pct_centinela
0,proveedor,numérica,0.0,1.0,0 (73504),61.89
1,categoria_ocupacion,numérica,0.0,7.0,0 (68654),57.81
2,ftp,numérica,0.0,1.0,0 (39371),33.15
3,no_prevision,numérica,0.0,1.0,0 (32705),27.54
4,cine11_1d,numérica,0.0,9.0,0 (23033),19.39
5,cae_general,numérica,0.0,9.0,0 (18321),15.43
6,est_conyugal,numérica,0.0,99.0,"0 (18321), 88 (2), 99 (1)",15.43
7,nivel,numérica,0.0,99.0,"0 (3566), 88 (3), 99 (1)",3.01
8,termino_nivel,numérica,1.0,99.0,"88 (6), 99 (2975)",2.51
9,meses_ocup_actual,numérica,0.0,9999.0,"0 (2309), 88 (124), 99 (122), 9999 (113)",2.25


In [ ]:
def _limpio(v):
    """Convierte escalares de NumPy a tipos nativos y quita el .0 de los enteros guardados como float."""
    if isinstance(v, (float, np.floating)) and float(v).is_integer():
        return int(v)
    if isinstance(v, np.generic):
        return v.item()
    return v


def inicio_fin(df, col, n=5):
    """Primeros y últimos valores únicos ordenados, sin tipos de NumPy."""
    vals = pd.Series(df[col].dropna().unique()).tolist()   # lista de tipos nativos de Python
    try:
        vals = sorted(vals, key=float)
    except (ValueError, TypeError):
        vals = sorted(vals, key=str)
    vals = [_limpio(v) for v in vals]
    return vals[:n], vals[-n:]


def mostrar_inicio_fin(df, excluir=EXCLUIR, n=5):
    for col in df.columns:
        if col in excluir:
            continue
        ini, fin = inicio_fin(df, col, n)
        print(f'{col}: {ini} ... {fin}')

In [ ]:
# Inicio y final de cada columna
mostrar_inicio_fin(df)

mes_central: [2, 5, 8, 11] ... [2, 5, 8, 11]
mes_encuesta: [1, 2, 3, 4, 5] ... [8, 9, 10, 11, 12]
fact_anual: ['1.62107250011471', '1.64546289032085', '1.6525510477332', '1.67113406012551', '1.67741505898929'] ... ['1141.5353096807', '1246.40684970563', '1306.56823102693', '1499.33372430408', '1626.33475387757']
conglomerado: [10004, 10008, 10038, 10073, 10127] ... [45804, 45805, 45822, 45841, 45857]
estrato: [1100111, 1100112, 1100113, 1100120, 1200110] ... [16100220, 16200110, 16200120, 16300110, 16300120]
activ: [1, 2, 3] ... [1, 2, 3]
region: [1, 2, 3, 4, 5] ... [12, 13, 14, 15, 16]
tipo: [1, 3] ... [1, 3]
sexo: [1, 2] ... [1, 2]
edad: [0, 1, 2, 3, 4] ... [100, 101, 102, 103, 104]
nacionalidad: [32, 56, 68, 76, 152] ... [804, 826, 840, 858, 862]
orig1: [1, 2, 88, 99] ... [1, 2, 88, 99]
est_conyugal: [0, 1, 2, 3, 4] ... [4, 5, 6, 88, 99]
estudia_actual: [1, 2, 88, 99] ... [1, 2, 88, 99]
parentesco: [1, 2, 3, 4, 5] ... [8, 9, 10, 11, 88]
proveedor: [0, 1] ... [0, 1]
nivel: [0, 1, 2, 

**Centinelas en las 30 columnas seleccionadas según la documentación**

El catálogo del INE documenta códigos especiales en 17 de las 30 columnas. Los más frecuentes son 88 (no sabe) y 99 (no responde). Otros códigos son específicos de cada variable: 77 (no aplica), 999 (sin información), 898 (zona no especificada) y 0 (no corresponde o no aplica). En las demás columnas no hay códigos especiales documentados, aunque algunas tienen vacíos, que deben interpretarse según el filtro de la pregunta.

| Columna | Códigos especiales documentados |
|---|---|
| `orig1` | 88 = no sabe; 99 = no responde |
| `est_conyugal` | 0 = no aplica; 88 = no sabe; 99 = no responde |
| `estudia_actual` | 88 = no sabe; 99 = no responde |
| `parentesco` | 88 = no sabe; 99 = no responde |
| `proveedor` | 0 = no corresponde |
| `nivel` | 0 = nunca estudió (valor válido); 88 = no sabe; 99 = no responde |
| `termino_nivel` | 88 = no sabe; 99 = no responde |
| `cine11_1d` | 0 = menos que primaria (valor válido); 9 = sin clasificar |
| `nacionalidad` | 898 = zona no especificada; 999 = sin información |
| `categoria_ocupacion` | 0 = no corresponde |
| `cae_general` | 0 = menor de 15 años (valor válido) |
| `ina_cal_edu` | 999 = sin información |
| `habituales` | 888 = no sabe; 999 = no responde; rango válido 1 a 168 |
| `no_prevision` | 0 = no corresponde |
| `plataformas_digitales` | 77 = no aplica; 88 = no sabe; 99 = no responde |
| `deseo_trabajar` | 88 = no sabe; 99 = no responde. Solo responden no ocupados |
| `ftp` | 0 = no corresponde |

Varias columnas tienen códigos que son valores válidos y no deben tratarse como centinelas. El 0 en `nivel`, `cine11_1d` y `cae_general` tiene significado propio. En `edad` el rango documentado es 0 a 120, así que los casos con 88 o 99 son edades posibles y deben revisarse caso por caso, no eliminarse automáticamente.

Hay también discrepancias entre el catálogo y el archivo. En `ftp` el catálogo define 0 como "no corresponde", pero en el panel los no inactivos aparecen como vacíos. En `habituales` la documentación indica que 888 y 999 son códigos de no respuesta, mientras que el rango válido llega hasta 168. Los valores 112, 140 y 168 caen dentro del rango, aunque sean horas semanales muy altas.

Dos columnas de la selección no tienen entrada en el catálogo: `idrph` y `estrato`. `fact_anual` aparece como "Numérico" sin códigos especiales, y su coma decimal debe convertirse antes de usarla.

Según lo revisado, no hay valores imposibles. Todos los mínimos y máximos caen dentro de los códigos documentados o de rangos posibles. Lo que sí distorsiona la lectura son los **códigos centinela** (88, 99 y similares), que aparecen como valores extremos en varias columnas, y el hecho de que el promedio y la desviación no significan nada en variables categóricas.

#### Revisión por columna

| Columna | Rango observado | Lectura |
|---|---|---|
| `mes_central` | 2 a 11 | Válido. Solo toma 2, 5, 8 y 11 |
| `mes_encuesta` | 1 a 12 | Válido |
| `region` | 1 a 16 | Válido |
| `tipo` | 1 a 3 | Válido. El código 2 no aparece en 2025, como habíamos visto |
| `sexo` | 1 a 2 | Válido |
| `edad` | 0 a 104 | Plausible. El 0 corresponde a menores de un año |
| `nacionalidad` | 32 a 862 | Códigos de país. El promedio no tiene sentido |
| `orig1` | 1 a 99 | 1 y 2 válidos. El 88 y el 99 son "no sabe" y "no responde" |
| `est_conyugal` | 0 a 99 | El 0 es "no aplica". El 99 es centinela |
| `estudia_actual` | 1 a 99 | El 88 y el 99 son centinelas |
| `parentesco` | 1 a 88 | El 88 es "no sabe" |
| `proveedor` | 0 a 1 | Válido |
| `nivel` | 0 a 99 | El 0 es "nunca estudió" (válido). El 99 es centinela |
| `termino_nivel` | 1 a 99 | El 88 y el 99 son centinelas |
| `cine11_1d` | 0 a 9 | El 9 es "sin clasificar" |
| `categoria_ocupacion` | 0 a 7 | Válido según el catálogo |
| `cae_general` | 0 a 9 | Revisar que el código 8 y 9 estén documentados en el catálogo |
| `activ` | 1 a 3 | Válido. Tiene 100.439 registros porque los menores de 15 años son nulos |

#### Lo que distorsiona la lectura

- **`termino_nivel`** tiene una desviación de 15,27. Con valores entre 1 y 2 normalmente, esa dispersión solo se explica por los códigos 88 y 99. La misma señal aparece en `nivel`, `estudia_actual` y `est_conyugal`.
- **`parentesco`** tiene un máximo de 88, que corresponde a "no sabe". El promedio de 3,03 está inflado por ese código.
- **`nacionalidad`**, **`estrato`**, **`conglomerado`** e **`idrph`** son identificadores o códigos. Su promedio y su desviación no tienen interpretación.

### 1.3 Consistencia — reglas cruzadas

### ✏️ Actividad 1.3

Escribe **dos reglas de consistencia** que tu conjunto debería cumplir y verifícalas. Ejemplos:

- «La fecha de término no puede ser anterior a la de inicio.»
- «Si el estado es *cerrado*, el monto vigente debe estar vacío.»
- «La suma de las partes debe igualar el total.»

Estas reglas no salen de los datos: salen de entender qué representan.

**La primera vincula la condición laboral con la categoría de ocupación. La segunda vincula la cotización previsional con el tipo de asalariado.**

### Regla 1: condición laboral y categoría de ocupación

**Regla:** si una persona está ocupada, tiene una categoría de ocupación válida (1 a 7) y tiene formalidad informada. Si está desocupada o inactiva, la categoría es 0 (no corresponde) y la formalidad está vacía.

**Por qué:** así lo documenta el catálogo. `categoria_ocupacion` define 0 como "no corresponde", y `ocup_form` existe solo para ocupados.

**Verificación:**
- Ocupados que cumplen la regla: 50.106 de 50.106.
- No ocupados que cumplen la regla: 50.333 de 50.333.
- Violaciones: 0.

### Regla 2: cotización previsional y asalariados

**Regla:** `no_prevision` debe tener respuesta si y solo si la persona es asalariada (categoría 3, 4, 5 o 6).

**Por qué:** la documentación define `no_prevision` como "persona ocupada dependiente cuyo empleador no paga cotizaciones", así que solo tiene sentido para asalariados.

**Verificación:**
- Asalariados sin `no_prevision`: 0.
- No asalariados con `no_prevision`: 0.
- Violaciones: 0.

**Pero hay un problema de interpretación.** Entre los asalariados, `no_prevision` vale 0 en 32.705 casos y 1 en 4.727. El catálogo define 0 como "no corresponde", pero entre asalariados el 0 significa que el empleador sí paga cotizaciones, no que la pregunta no aplique. Un analista que lea solo el código podría concluir que esas personas no tienen cotización, cuando en realidad sí la tienen. Conviene documentar el 0 como "sí cotiza" en este caso, y consultarlo con el INE.

In [ ]:
## Código de verificación

categoria_n = pd.to_numeric(df['categoria_ocupacion'], errors='coerce')
activ_n = pd.to_numeric(df['activ'], errors='coerce')

# Regla 1:
ocupado = activ_n == 1
viol1 = (ocupado & ~categoria_n.between(1, 7)) | \
        (~ocupado & ((categoria_n != 0) | df['ocup_form'].notna()))

# Regla 2:
asalariado = categoria_n.isin([3, 4, 5, 6])
viol2 = (asalariado & df['no_prevision'].isna()) | (~asalariado & df['no_prevision'].notna())

print('Violaciones regla 1:', int(viol1.sum()))
print('Violaciones regla 2:', int(viol2.sum()))

Violaciones regla 1: 0
Violaciones regla 2: 0


### 1.4 Unicidad

No existe una sola columna que identifique cada fila. `idrph` identifica a la persona, pero cada persona aparece cuatro veces en el panel. Cada fila es una visita de una persona, y la clave que la identifica es la combinación de dos campos:

**`idrph` + `mes_central`**

La combinación es única en las 118.760 filas, sin duplicados. `mes_central` indica el trimestre móvil de la visita (2, 5, 8 u 11). Como cada persona tiene exactamente una visita por trimestre, esta combinación funciona como clave.

In [ ]:
print('Duplicados exactos:', df.duplicated().sum())

# Si tu conjunto tiene una columna identificadora, revísala también:
print('Duplicados por idrph y mes_central:', df.duplicated(['idrph', 'mes_central']).sum())   # debe dar 0

Duplicados exactos: 0
Duplicados por idrph y mes_central: 0


### ✏️ Actividad 1.4

Construye una **clave normalizada** (sin espacios, en mayúsculas, sin tildes) con las columnas que deberían identificar un registro, y cuenta los casi-duplicados.

In [ ]:
def normalizar_texto(serie):
    return (serie.astype(str).str.strip().str.upper()
                 .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))

# Tu código aquí

df['clave_normalizada'] = normalizar_texto(df['idrph'] + df['mes_central']) # bitácora

In [ ]:
def normalizar(x):
    """Clave normalizada: sin espacios sobrantes, en mayúsculas, sin tildes,
    con coma decimal convertida a punto y enteros sin '.0'."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = s.replace(',', '.') if re.fullmatch(r'-?\d+,\d+', s) else s
    try:
        f = float(s)
        return str(int(f)) if f.is_integer() else str(f)
    except ValueError:
        pass
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', s).strip().upper()


def tipos_de_variacion(raw, norm=None):
    """Describe en qué difiere el valor original de una escritura estándar."""
    tipos = []
    s = str(raw)
    if s != s.strip() or re.search(r'\s{2,}', s):
        tipos.append('espacios')
    if re.fullmatch(r'-?\d+,\d+', s.strip()):
        tipos.append('coma decimal')
    elif re.fullmatch(r'-?\d+\.0+', s.strip()):
        tipos.append('sufijo .0')
    if s != s.upper() and not re.fullmatch(r'[\d.,\s-]+', s):
        tipos.append('capitalización')
    if any(unicodedata.combining(ch) for ch in unicodedata.normalize('NFKD', s)):
        tipos.append('tildes')
    return ', '.join(tipos) or 'idéntico'


def candidatos_casi_duplicados(df, columnas=None, clave=None, excluir=('idrph',)):
    """
    Busca casi-duplicados en dos niveles:
      - valor: distintas escrituras de un mismo valor dentro de una columna.
      - registro: filas con la misma clave normalizada pero texto original distinto.
    Devuelve un dataframe con un caso candidato por fila.
    """
    columnas = columnas or [c for c in df.columns if c not in excluir]
    filas = []

    # Nivel valor
    for col in columnas:
        s = df[col].dropna()
        if s.empty:
            continue
        tabla = pd.DataFrame({'original': s.astype(str), 'clave': s.map(normalizar)})
        conteo = tabla.groupby('clave')['original'].nunique()
        for clave_norm in conteo[conteo > 1].index:
            sub = tabla[tabla['clave'] == clave_norm]
            for original, n in sub['original'].value_counts().items():
                filas.append({
                    'nivel': 'valor',
                    'columna': col,
                    'clave_normalizada': clave_norm,
                    'valor_original': original,
                    'casos': int(n),
                    'variantes_en_grupo': int(sub['original'].nunique()),
                    'tipo_variacion': tipos_de_variacion(original),
                })

    # Nivel registro
    if clave:
        clave_norm = df[clave].apply(lambda r: ' | '.join(str(normalizar(v)) for v in r), axis=1)
        original = df[clave].apply(lambda r: ' | '.join(str(v) for v in r), axis=1)
        tabla = pd.DataFrame({'clave_norm': clave_norm, 'original': original})
        conteo = tabla.groupby('clave_norm')['original'].nunique()
        for k in conteo[conteo > 1].index:
            sub = tabla[tabla['clave_norm'] == k]
            for orig, n in sub['original'].value_counts().items():
                filas.append({
                    'nivel': 'registro',
                    'columna': ' + '.join(clave),
                    'clave_normalizada': k,
                    'valor_original': orig,
                    'casos': int(n),
                    'variantes_en_grupo': int(sub['original'].nunique()),
                    'tipo_variacion': 'clave con variantes de formato',
                })

    cols_salida = ['nivel', 'columna', 'clave_normalizada', 'valor_original',
                   'casos', 'variantes_en_grupo', 'tipo_variacion']
    resumen = pd.DataFrame(filas, columns=cols_salida)
    return resumen.sort_values(['nivel', 'columna', 'clave_normalizada']).reset_index(drop=True)

In [ ]:
df.columns

Index(['idrph', 'mes_central', 'mes_encuesta', 'fact_anual', 'conglomerado',
       'estrato', 'activ', 'region', 'tipo', 'sexo', 'edad', 'nacionalidad',
       'orig1', 'est_conyugal', 'estudia_actual', 'parentesco', 'proveedor',
       'nivel', 'termino_nivel', 'cine11_1d', 'categoria_ocupacion',
       'cae_general', 'ocup_form', 'ina_cal_edu', 'habituales',
       'meses_ocup_actual', 'no_prevision', 'plataformas_digitales',
       'deseo_trabajar', 'ftp', 'clave_normalizada'],
      dtype='object')

In [ ]:
# ---------- Uso ----------
# Nivel valor en columnas de texto
casos_valor = candidatos_casi_duplicados(df, columnas=['region', 'nacionalidad'])
casos_valor

,nivel,columna,clave_normalizada,valor_original,casos,variantes_en_grupo,tipo_variacion


### 1.5 Oportunidad

In [ ]:
# Si tienes alguna columna de fecha, ¿cuál es el registro más reciente?

# mes_encuesta más reciente
print("Mes encuesta más reciente: {}".format(df['mes_encuesta'].max()))
print()

print("¿El conjunto está vigente para la pregunta que quieres responder?")
print("R: Sí, porque la descarga era del año 2025 y se validó anteriormente que son casos de las personas que recibieron 4 visitas en el mes siendo el último uno del último trimestre.")


Mes encuesta más reciente: 12

¿El conjunto está vigente para la pregunta que quieres responder?
R: Sí, porque la descarga era del año 2025 y se validó anteriormente que son casos de las personas que recibieron 4 visitas en el mes siendo el último uno del último trimestre.


---
## Parte 2 · Limpieza, decisión por decisión

Ahora corriges. **Regla del laboratorio: cada celda de esta parte tiene su fila en la bitácora.** Si aplicas una transformación que no registraste, no cuenta.

Recuerda las tres decisiones legítimas: **Corregir**, **Descartar** o **Declarar como limitación**.

### ✏️ Actividad 2.1 — Centinelas a faltantes

In [ ]:
cols_centinelas = ['orig1', 'est_conyugal', 'estudia_actual', 'parentesco',
                       'proveedor', 'nivel', 'termino_nivel', 'cine11_1d',
                       'categoria_ocupacion', 'cae_general', 'ina_cal_edu',
                       'habituales', 'meses_ocup_actual', 'no_prevision', 'plataformas_digitales',
                       'deseo_trabajar', 'ftp']
mostrar_inicio_fin(df[cols_centinelas])

orig1: [1, 2, 88, 99] ... [1, 2, 88, 99]
est_conyugal: [0, 1, 2, 3, 4] ... [4, 5, 6, 88, 99]
estudia_actual: [1, 2, 88, 99] ... [1, 2, 88, 99]
parentesco: [1, 2, 3, 4, 5] ... [8, 9, 10, 11, 88]
proveedor: [0, 1] ... [0, 1]
nivel: [0, 1, 2, 3, 4] ... [12, 13, 14, 88, 99]
termino_nivel: [1, 2, 88, 99] ... [1, 2, 88, 99]
cine11_1d: [0, 1, 2, 3, 4] ... [3, 4, 5, 6, 9]
categoria_ocupacion: [0, 1, 2, 3, 4] ... [3, 4, 5, 6, 7]
cae_general: [0, 1, 2, 3, 4] ... [5, 6, 7, 8, 9]
ina_cal_edu: [1, 2, 3, 999] ... [1, 2, 3, 999]
habituales: [1, 2, 3, 4, 5] ... [112, 140, 168, 888, 999]
meses_ocup_actual: ['0', '1', '1,5', '10', '100'] ... ['96', '97', '98', '99', '9999']
no_prevision: [0, 1] ... [0, 1]
plataformas_digitales: [1, 2, 88, 99] ... [1, 2, 88, 99]
deseo_trabajar: [1, 2, 88, 99] ... [1, 2, 88, 99]
ftp: [0, 1] ... [0, 1]


In [ ]:
df[cols_centinelas].dtypes

,0
orig1,float64
est_conyugal,int64
estudia_actual,int64
parentesco,int64
proveedor,int64
nivel,int64
termino_nivel,int64
cine11_1d,int64
categoria_ocupacion,int64
cae_general,int64


In [ ]:
# centinelas = [88, 99, 888, 999, 9999]

# # for col in cols_centinelas:
# #     df[col] = df[col].replace(centinelas, pd.NA)
# #     mostrar_inicio_fin(df[[col]])

# for col in cols_centinelas: # bitacora
#     s = pd.to_numeric(df[col], errors='coerce')
#     df[col] = s.mask(s.isin(centinelas)).astype('Int64')

centinelas_por_columna = {
    'orig1': [88, 99],
    'est_conyugal': [88, 99],
    'estudia_actual': [88, 99],
    'parentesco': [88, 99],
    'nivel': [88, 99],
    'termino_nivel': [88, 99],
    'cine11_1d': [88, 99],
    'categoria_ocupacion': [88, 99],
    'cae_general': [88, 99],
    'ina_cal_edu': [999],
    'habituales': [888, 999],
    'meses_ocup_actual': [9999],
    'no_prevision': [99],
    'plataformas_digitales': [88, 99],
    'deseo_trabajar': [88, 99],
    'ftp': [99],

    # ... resto de columnas con sus códigos documentados
}

for col, codigos in centinelas_por_columna.items():
    s = pd.to_numeric(df[col], errors='coerce')
    df[col] = s.mask(s.isin(codigos)).astype('Int64')

### ✏️ Actividad 2.2 — Normalización de categorías

Normaliza mecánicamente y después revisa lo que queda: los plurales, las abreviaturas y los sinónimos necesitan un diccionario de equivalencias que escribes tú.

In [ ]:
# Identificadores: texto. No se operan aritméticamente.
cols_naturaleza_text = ['idrph', 'conglomerado', 'estrato']

# Categorías: códigos con etiqueta, sin orden implícito.
cols_naturaleza_category = [
    'activ', 'region', 'tipo', 'sexo', 'nacionalidad', 'orig1', 'est_conyugal',
    'estudia_actual', 'parentesco', 'proveedor', 'termino_nivel',
    'categoria_ocupacion', 'cae_general', 'ocup_form', 'ina_cal_edu',
    'no_prevision', 'plataformas_digitales', 'deseo_trabajar', 'ftp',
]

# Enteros con orden: categorías ordenables, como enteros nulables.
cols_naturaleza_int = ['mes_central', 'mes_encuesta', 'nivel', 'cine11_1d']

# Cantidades reales: numéricas continuas o contables.
cols_naturaleza_float = ['fact_anual', 'edad', 'habituales', 'meses_ocup_actual']

In [ ]:
# normalizar valores de las columnas del dataframe

# for col in cols_naturaleza_text + cols_naturaleza_category:
#     df[col] = normalizar_texto(df[col])
# df.dtypes

def aplicar_tipos_por_lista(df):
    out = df.copy()

    for col in cols_naturaleza_text:
        out[col] = out[col].astype('string')

    for col in cols_naturaleza_category:
        out[col] = (pd.to_numeric(out[col], errors='coerce')
                      .astype('Int64').astype('string').astype('category'))

    for col in cols_naturaleza_int:
        out[col] = pd.to_numeric(out[col], errors='coerce').astype('Int64')

    for col in cols_naturaleza_float:
        out[col] = (out[col].astype(str).str.replace(',', '.', regex=False)
                            .pipe(pd.to_numeric, errors='coerce').astype('float64'))

    return out

df = aplicar_tipos_por_lista(df) # bitacora

La función `normalizar_texto` no hace falta para las categorías. Para los identificadores tampoco es necesaria, porque `idrph`, `conglomerado` y `estrato` son solo dígitos y no tienen mayúsculas ni tildes.

In [ ]:
for col in cols_naturaleza_text + cols_naturaleza_category:
    print(f"Valores únicos en la columna {col}:")
    print(df[col].unique())
    print()

Valores únicos en la columna idrph:
<StringArray>
['966906226666', '246247268292', '254443118268', '553075369226',
 '724298626488', '255735824251',  '60893230467', '277282968944',
 '136336296741', '528849653300',
 ...
 '987404100267', '282833616327', '334228610107', '662797938418',
 '619556936147', '319590634157', '359967663144', '334571800522',
 '267636369515', '624348254665']
Length: 29690, dtype: string

Valores únicos en la columna conglomerado:
<StringArray>
['20043', '36033', '14530', '13536', '31039', '39309', '13326', '11146',
 '34193', '21498',
 ...
 '25316', '36411', '27021', '29987', '10127', '21259', '26325', '33925',
 '41365', '19282']
Length: 2621, dtype: string

Valores únicos en la columna estrato:
<StringArray>
['13100312', '13100313',  '3200110',  '8100312',  '8100212', '13100112',
  '6300121',  '8100112',  '8100313',  '8100311',
 ...
  '7200120',  '8100213',  '5100320',  '4300210', '13500111', '13600120',
  '3300120', '13200111', '13300113', '12000220']
Length: 232, 

### ✏️ Actividad 2.3 — Duplicados

In [ ]:
# Tu código aquí.

# No existen duplicadoss según se ve en el apartado 1.4

### ✏️ Actividad 2.4 — Tipos de dato

Convierte a numérico lo que deba ser numérico y a fecha lo que deba ser fecha. Si hay formatos mixtos, prueba formatos explícitos en orden en vez de dejar que pandas adivine.

In [ ]:
# Tu código aquí

# Fue verificado en el apartado 2.2

df.dtypes

,0
idrph,string[python]
mes_central,Int64
mes_encuesta,Int64
fact_anual,float64
conglomerado,string[python]
estrato,string[python]
activ,category
region,category
tipo,category
sexo,category


### ✏️ Actividad 2.5 — Atípicos

Identifica los valores extremos y **pronúnciate sobre cada grupo**: ¿error de registro o fenómeno real?

Cruza la variable con otra que te permita juzgar plausibilidad, como en el Ejercicio 8. No elimines atípicos solo por serlo.

In [ ]:
# Tu código aquí

for c in cols_naturaleza_float:
    s = df[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite = q3 + 1.5 * iqr
    print(f'''{c}:
    Q1={q1:.0f} - Q3={q3:.0f} - IQR={iqr:.0f} - límite={limite:.0f} - máximo={s.max():.0f} - atípicos={(s > limite).sum()}''')

fact_anual: 
    Q1=24 - Q3=53 - IQR=29 - límite=96 - máximo=1626 - atípicos=10058
edad: 
    Q1=22 - Q3=62 - IQR=40 - límite=122 - máximo=104 - atípicos=0
habituales: 
    Q1=35 - Q3=44 - IQR=9 - límite=58 - máximo=168 - atípicos=2303
meses_ocup_actual: 
    Q1=12 - Q3=130 - IQR=118 - límite=307 - máximo=1016 - atípicos=3393


In [ ]:
# conteo de cuántos casos caen en las reglas de plausibilidad y registra ese número con su evidencia. La decisión depende de ese conteo, no del IQR.

print('habituales > 100 h/sem:', int((df['habituales'] > 100).sum()))
print('meses_ocup_actual > (edad - 15) * 12:',
      int((df['meses_ocup_actual'] > (df['edad'] - 15) * 12).sum()))

habituales > 100 h/sem: 31
meses_ocup_actual > (edad - 15) * 12: 226


In [ ]:
m = df['meses_ocup_actual']
edad_max_meses = (df['edad'] - 15) * 12

CENTINELAS_MESES = [9999]          # códigos confirmados (inferidos de b17_ano)
sin_centinela = ~m.isin(CENTINELAS_MESES)

violaciones     = (m > edad_max_meses) & sin_centinela
violaciones_tol = (m > edad_max_meses + 12) & sin_centinela
print('Violaciones:', int(violaciones.sum()))
print('Violaciones con tolerancia de 12 meses:', int(violaciones_tol.sum()))

Violaciones: 226
Violaciones con tolerancia de 12 meses: 131


_Tu conclusión sobre los atípicos:_

- **Violaciones:** 226, que incluyen el valor de 1016 meses.
- **Violaciones con tolerancia de 12 meses:** 131.

**Filas definitivas para la bitácora**

| Columna | Defecto | Evidencia | Decisión | Justificación | Efecto esperado |
|---|---|---|---|---|---|
| `meses_ocup_actual` | Duración superior al máximo posible por edad | 131 casos que superan `(edad − 15) × 12` en más de 12 meses. Otros 95 quedan dentro de esa tolerancia. Total de violaciones: 226 | Declarar | Probablemente errores de registro o de unidad, sin forma de confirmarlo. No se eliminan | Medias de duración potencialmente infladas. Reportar con y sin estos casos |
| `habituales` | Horas por encima de 100 | 31 casos entre 101 y 168, todos ocupados | Declarar | Dentro del rango documentado (1 a 168), sin evidencia de error | Puede subir la media de horas. Reportar con y sin estos casos |

---
## Parte 3 · Transformación y nuevos atributos

Preparar no es solo corregir: también es **construir** lo que el análisis va a necesitar.

### ✏️ Actividad 3.1 — Un atributo derivado

Crea **al menos una variable nueva** a partir de las existentes, que aporte a tu pregunta analítica. Ejemplos: una razón entre dos columnas, la antigüedad en años a partir de una fecha, un tramo a partir de un valor continuo.

Explica en la celda siguiente por qué esa variable ayuda a responder tu pregunta.

####  1. Nivel educativo agrupado y flag de superior completa

Estas dos columnas resumen el nivel educativo de cada persona en formas que el análisis puede usar directamente.

**`nivel_grupo`** agrupa los 14 códigos de nivel educacional en tres categorías: básica o menos, media y superior. El catálogo describe 14 niveles distintos, demasiados para comparar grupos con pocos casos en cada uno. Con tres categorías, las diferencias entre personas se leen con claridad y los grupos son lo bastante grandes para estimar sus tasas de desocupación con precisión. El código 14 (normalista) se asignó a media, porque el catálogo no lo clasifica y así se define en la bitácora.

**`superior_completa`** es una variable binaria: vale 1 cuando la persona tiene un nivel superior (técnico, profesional, universitario o de posgrado) y terminó ese nivel, y 0 en el resto de los casos. Separa a quienes completaron una formación superior de quienes no, y es la comparación que la pregunta de factores asociados a la desocupación necesita.

Ambas son importantes porque la educación es uno de los factores que más se asocia con la desocupación, y estas columnas permiten medirla sin perder información sobre los niveles intermedios. Además, ayudan a interpretar los resultados: un modelo que use `nivel_grupo` o `superior_completa` es más fácil de explicar que uno con 14 categorías.

In [ ]:
# bitacora
# Se extraen las columnas 'nivel' y 'termino_nivel' para facilitar la manipulación
# Se crean variables auxiliares para simplificar la lógica de creación de nuevas columnas

n = df['nivel'].astype('float64')
t = df['termino_nivel'].astype('float64')

# variable nivel_grupo como categoría según el rango de nivel
# Básica o menos: 0-3
# Media: 4-6 y 14
# Superior: 7-13
df['nivel_grupo'] = pd.Series(
    np.select(
        [n.between(0, 3),
         n.between(4, 6) | (n == 14),
         n.between(7, 13)],
        ['Básica o menos', 'Media', 'Superior'],
        default=None),
    index=df.index).astype('category')

# variable superior_completa indica si el nivel es superior y se completó el nivel
# 1 si es superior y se completó, 0 en caso contrario
df['superior_completa'] = (n.between(7, 13) & (t == 1)).astype(int)

# Verificación
print('Vacíos en nivel:', int(df['nivel'].isna().sum()))
print('Vacíos en nivel_grupo:', int(df['nivel_grupo'].isna().sum()))
print(df['nivel_grupo'].value_counts(dropna=False))

Vacíos en nivel: 4
Vacíos en nivel_grupo: 4
nivel_grupo
Media             44120
Básica o menos    42238
Superior          32398
NaN                   4
Name: count, dtype: int64


In [ ]:
# df con las variables nuevas y las que permiten crear dichas variables
df[['nivel', 'termino_nivel', 'nivel_grupo', 'superior_completa']]

,nivel,termino_nivel,nivel_grupo,superior_completa
0,5,1,Media,0
1,12,1,Superior,1
2,8,1,Superior,1
3,0,<NA>,Básica o menos,0
4,7,1,Superior,1
...,...,...,...,...
118755,3,2,Básica o menos,0
118756,3,2,Básica o menos,0
118757,4,1,Media,0
118758,3,2,Básica o menos,0


### ✏️ Actividad 3.2 — Normalización o codificación

Si tu conjunto tiene variables numéricas en escalas muy distintas, o categóricas que necesitarás codificar en la semana 4, déjalas preparadas aquí.

In [ ]:
# Tu código aquí. Referencia:
# from sklearn.preprocessing import StandardScaler, MinMaxScaler
# df_cod = pd.get_dummies(df, columns=['TU_COLUMNA_CATEGORICA'], drop_first=True)

# BITACORA: declarar acción pendiente hasta seguir evaluando la necesidad de nuevas variables y quizás la remoción de algunas existentes

---
## Parte 4 · El flujo como función reutilizable

Junta todo lo anterior en **una sola función**. Esto es lo que evalúa el RA3 y lo que se revisa en la Sección A de la Entrega Final: que tu limpieza se pueda volver a ejecutar de principio a fin.

In [ ]:
def limpiar(ruta, sep=SEPARADOR, semilla=SEMILLA):
    """Flujo completo de limpieza de mi conjunto. Devuelve (df_limpio, informe)."""
    np.random.seed(semilla)
    d = pd.read_csv(ruta, sep=sep, encoding='utf-8', low_memory=False,
                    dtype={'fact_anual': str})
    informe = {'filas_iniciales': len(d), 'columnas_iniciales': d.shape[1]}

    # 0 · subconjunto: panel de personas con exactamente 4 visitas
    visitas = d.groupby('idrph')['idrph'].transform('size')
    d = d[visitas == 4].copy()
    informe['filas_panel'] = len(d)
    informe['personas_panel'] = int(d['idrph'].nunique())

    # selección de las 30 columnas de análisis
    d = d[cols_analisis].copy()

    # 1 · centinelas -> faltantes (Int64 nulable)
    for col in cols_centinelas:
        s = pd.to_numeric(d[col], errors='coerce')
        d[col] = s.mask(s.isin(centinelas)).astype('Int64')

    # 2 · categorías y 4 · tipos: se resuelven juntos al tipar cada columna
    d = aplicar_tipos_por_lista(d)

    # 3 · duplicados: el diagnóstico (1.4) no encontró duplicados, no se elimina nada
    # 5 · atípicos: solo se diagnosticaron (2.5); no se modifican valores

    # 6 · atributos derivados
    n = d['nivel'].astype('float64')
    t = d['termino_nivel'].astype('float64')
    d['nivel_grupo'] = pd.Series(
        np.select([n.between(0, 3),
                   n.between(4, 6) | (n == 14),
                   n.between(7, 13)],
                  ['Básica o menos', 'Media', 'Superior'],
                  default=None),
        index=d.index).astype('category')
    d['superior_completa'] = (n.between(7, 13) & (t == 1)).astype(int)

    informe['filas_finales'] = len(d)
    informe['columnas_finales'] = d.shape[1]
    return d, informe


limpio, informe = limpiar(RUTA + ARCHIVO)
print(informe)

{'filas_iniciales': 390903, 'columnas_iniciales': 222, 'filas_panel': 118760, 'personas_panel': 29690, 'filas_finales': 118760, 'columnas_finales': 32}


In [ ]:
# Comprobación: ejecuta la función DOS VECES y verifica que da lo mismo.
a, _ = limpiar(RUTA + ARCHIVO)
b, _ = limpiar(RUTA + ARCHIVO)
print('¿Reproducible?', a.equals(b))

¿Reproducible? True


In [ ]:
# Guarda el conjunto limpio. Este archivo es el insumo de la semana 3.

RUTA_semana2 = '/content/drive/Othercomputers/Notebook HP/Ing Informática UCSC/Trimestre 3/1 Ciencia de datos/Evaluaciones/Semana 2/Lab2/'

if not os.path.exists(RUTA_semana2 + 'mi_conjunto_limpio.csv'):
    limpio.to_csv(RUTA_semana2 +'mi_conjunto_limpio.csv', sep=SEPARADOR, index=False, encoding='utf-8')
    print('Archivo guardado: mi_conjunto_limpio.csv en', RUTA_semana2, limpio.shape)
else:
    print('Archivo no guardado: mi_conjunto_limpio.csv ya existe en', RUTA_semana2, limpio.shape)

Archivo guardado: mi_conjunto_limpio.csv en /content/drive/Othercomputers/Notebook HP/Ing Informática UCSC/Trimestre 3/1 Ciencia de datos/Evaluaciones/Semana 2/Lab2/ (118760, 32)


---
## Parte 5 · Versionado

Sube el cuaderno y el conjunto limpio a tu repositorio. Un commit con mensaje descriptivo, no «cambios».

In [ ]:
# En una terminal:
# git add S2_Notebook_Laboratorio_CDD.ipynb mi_conjunto_limpio.csv
# git commit -m 'Semana 2: diagnostico de calidad y flujo de limpieza'
# git push

**Enlace a mi repositorio:** [https://github.com/hersil7/cdd_ucsc](https://github.com/hersil7/cdd_ucsc)

---

## Antes de cerrar el laboratorio

- [ ] Las cinco dimensiones diagnosticadas y registradas en la hoja «Resumen de calidad»
- [ ] Cada transformación aplicada tiene su fila en la bitácora, con justificación
- [ ] Me pronuncié sobre los atípicos: cuáles son error y cuáles fenómeno real
- [ ] Creé al menos un atributo derivado y expliqué para qué sirve
- [ ] La función `limpiar()` corre de principio a fin y entrega lo mismo dos veces
- [ ] El conjunto limpio está guardado y versionado
- [ ] Tengo anotadas las limitaciones que declararé en el informe

Lo que quede pendiente, al Foro de Consultas o a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 2_